## RAG Pipeline - Data Ingestion to Vector DB Pipeline 

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\manas\AppData\Local\Temp\ipykernel_32912\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\manas\Documents\development\langgraph-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### read all the pdfs inside the directory 
def process_all_pdfs(pdf_directory): 
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively 
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to precess")

    for pdf_file in pdf_files: 
        print(f"\nProcessing: {pdf_file.name}")
        try: 
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to meta data 
            for doc in documents: 
                doc.metadata['source_file'] = pdf_file.name 
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e: 
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#process all PDFs in the data diretory 
all_pdf_documents = process_all_pdfs("./data")



Found 2 PDF files to precess

Processing: attention.pdf
 Loaded 15 pages

Processing: WordEmbeddingsinNLP.pdf
 Loaded 18 pages

Total documents loaded: 33


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toront

In [ ]:
### Text splitting get into chunks 

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200): 
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, 
        chunk_overlap = chunk_overlap, 
        length_function = len, 
        separators=["\n\n", "\n", " ", ""] # this makes sure that the chunks that you separate are not at random but is from paragraph, then sentences , then spaces and at last resort raw text
        
    )

    split_docs = text_splitter.split_documents(documents) # this is not a recursive function it is calling a different function (not the split_documents funciton that we created)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks") 

    # show example of a chunk 
    if split_docs: 
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs 

In [5]:
chunks= split_documents(all_pdf_documents) 
chunks

Split 33 documents into 87 chunks

Example chunk:
Content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toront

### Embeddings and Vector Database 


In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
class EmbeddingManager: 
    """Handles document embeddings generation using Sentence Transformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): 
        """
        Initialize the embedding manager 

        Args: 
            model_name : HuggingFace model name for sentence embeddings 
        """

        self.model_name =model_name 
        self.model = None 
        self._load_model()


    def _load_model(self): 
        """Load the Sentence Transformer model """
        try: 
            print(f"Loading embeddings model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e: 
            print(f"Error loading model {self.model_name}: {e}")
            raise ValueError("Model not loaded")



    def generate_embeddings(self, texts: List[str])-> np.ndarray: 
        """
        Generate embeddings for a list of texts 

        Args: 
            texts: List of text strings to embed 

        Returns: 
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model: 
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings



In [21]:
## initialize the embedding manager 
embedding_manager = EmbeddingManager()
embedding_manager

Loading embeddings model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14801.57it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store 

In [ ]:
class VectorStore: 
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./data/vector_store"): 
        """
        Initialize the vector store 

        Args: 
            collection_name: Name of the ChromaDB collection 
            persist_directory: Directory to presist the vector store 
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None 
        self.collection = None
        self._initialize_store()

    def _initialize_store(self): 
        """Initialize ChromaDB client and collection"""

        try: 
            # Create persistent ChormaDB client 
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or create collection 
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name, 
                metadata ={"description": "PDF document embeddings for RAG "}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e: 
            print(f"Error initializing vector store: {e}")
            raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray): 
        """
        Add documents and their embeddings to the vector store 

        Args: 
            documents; List of Langchain documents 
            embeddings: Corresponding embeddings for the documents 
        """

        if len(documents) != len(embeddings): 
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # prepare data for chromaDB 
        ids = [] 
        metadatas = []
        documents_text =[]
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)): 
            # Generate unique ID 
            doc_id = f"doc_{uuid.uuid4().hox[:8]}_{i}"
            ids.append(doc_id)


            # Prepare metadata 
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i 
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content 
            documents_text.append(doc.page_content)

            #Embedding 
            embeddings_list.append(embedding.tolist())

        # add to collection 
        try: 
            self.collection.add(
                ids = ids, 
                embeddings= embeddings_list, 
                metadatas = metadatas, 
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e: 
            print(f"Error adding documents to vector store: {e}")
            raise 
        



In [23]:
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0
